# Part 5: Association Rule Learning

**Quick Reference for Market Basket Analysis**

[Back to Index](Index.ipynb)

---
## 5.1 Apriori Algorithm

**Concept:** Find frequent itemsets and generate association rules

**Key Metrics:**
- **Support:** P(A) = frequency of itemset
- **Confidence:** P(B|A) = support(A∪B) / support(A)
- **Lift:** confidence(A→B) / support(B)

**Use Case:** Market basket analysis, recommendation systems

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# Example transaction data
transactions = [
    ['milk', 'bread', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread', 'butter', 'cheese'],
    ['milk', 'bread'],
    ['bread', 'butter']
]

# Transform to one-hot encoded DataFrame
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)
print(df.head())

# Find frequent itemsets
frequent_itemsets = apriori(df, min_support=0.4, use_colnames=True)
print(f"\nFrequent Itemsets:\n{frequent_itemsets}")

# Generate association rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
rules = rules.sort_values('lift', ascending=False)
print(f"\nAssociation Rules:\n{rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]}")

# Interpretation:
# Support: How often itemset appears
# Confidence: How often rule is correct
# Lift > 1: Items are positively correlated
# Lift = 1: Independent
# Lift < 1: Negatively correlated

### From DataFrame format

In [ ]:
# If data is in format: TransactionID, Items
# Convert to one-hot encoded format
basket = df.groupby(['TransactionID', 'Item'])['Item'].count().unstack().fillna(0)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

# Apply Apriori
frequent_itemsets = apriori(basket, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
print(rules.sort_values('lift', ascending=False).head(10))

---
## 5.2 FP-Growth Algorithm

**Concept:** Faster than Apriori, uses FP-tree structure

**Advantage:** No candidate generation, more efficient

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

# Same data format as Apriori
frequent_itemsets = fpgrowth(df, min_support=0.4, use_colnames=True)
print(f"Frequent Itemsets (FP-Growth):\n{frequent_itemsets}")

# Generate rules (same as Apriori)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
print(f"\nRules:\n{rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]}")

---
### Rule Filtering and Analysis

In [ ]:
# Filter rules by multiple criteria
filtered_rules = rules[
    (rules['lift'] > 1.2) & 
    (rules['confidence'] > 0.7) & 
    (rules['support'] > 0.05)
]

# Find rules with specific item
rules_with_item = rules[rules['antecedents'].apply(lambda x: 'milk' in x)]

# Visualize top rules
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(data=rules, x='support', y='confidence', size='lift', sizes=(50, 400), alpha=0.6)
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules')
plt.show()

# Heatmap
pivot = rules.pivot(index='antecedents', columns='consequents', values='lift')
plt.figure(figsize=(10, 8))
sns.heatmap(pivot, annot=True, cmap='coolwarm')
plt.title('Lift Values Heatmap')
plt.show()

---
### Practical Example: Retail Recommendations

In [ ]:
# Load retail data
# Format: TransactionID, ProductID/ProductName

# 1. Prepare data
basket = (
    retail_df
    .groupby(['TransactionID', 'Product'])['Quantity']
    .sum()
    .unstack()
    .fillna(0)
)
basket = (basket > 0).astype(int)  # Convert to binary

# 2. Find patterns
frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)

# 3. Top recommendations
top_rules = rules.sort_values('lift', ascending=False).head(20)

# 4. Create recommendation function
def recommend_products(purchased_items, rules_df, top_n=5):
    recommendations = []
    for item in purchased_items:
        matching_rules = rules_df[rules_df['antecedents'].apply(lambda x: item in x)]
        for _, rule in matching_rules.iterrows():
            for consequent in rule['consequents']:
                if consequent not in purchased_items:
                    recommendations.append((consequent, rule['confidence'], rule['lift']))
    
    # Sort and deduplicate
    recommendations = sorted(set(recommendations), key=lambda x: (x[2], x[1]), reverse=True)
    return recommendations[:top_n]

# Example usage
cart = ['bread', 'butter']
suggestions = recommend_products(cart, rules)
print(f"Customers who bought {cart} also bought: {suggestions}")

---
### Key Takeaways

**When to use:**
- Retail (market basket analysis)
- E-commerce (product recommendations)
- Web usage mining
- Healthcare (symptom-disease associations)

**Parameter Selection:**
- **min_support:** Start with 0.01-0.05 (1-5%), adjust based on data
- **min_confidence:** Typically 0.5-0.7 (50-70%)
- **min_lift:** > 1.0 for positive associations

**Apriori vs FP-Growth:**
- Apriori: Simple, easier to understand
- FP-Growth: Faster, better for large datasets

**Interpretation:**
- High support + high confidence = strong, frequent rule
- High lift = strong association (more likely to buy together)
- Low support but high lift = niche but strong association